# COLM Fiction Chunks — Qualitative Validation

Validates the 10 fiction novels fetched and chunked for the COLM norm extraction pipeline.
Checks: all books present, chunk sizes, metadata enrichment, first/last/middle chunk quality.

In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.getcwd()), '.env'))
CHUNKS_PATH = os.environ.get('FICTION_CHUNKS_PATH',
    '/share/pierson/matt/UAIR/outputs/2026-03-20_historical_norms/22-25-01/COLM_fetch_fiction/outputs/fetch/chunks.parquet')
df = pd.read_parquet(CHUNKS_PATH)
print(f'Loaded {len(df)} chunks from {CHUNKS_PATH}')
df.head(2)

## 1. Book inventory — are all 10 novels present?

In [ ]:
EXPECTED_BOOKS = {
    '1342': 'Pride and Prejudice',
    '11': "Alice's Adventures in Wonderland",
    '145': 'Middlemarch',
    '541': 'The Age of Innocence',
    '1023': 'Bleak House',
    '135': 'Les Misérables',
    '1399': 'Anna Karenina',
    '4078': 'The Picture of Dorian Gray',
    '1184': 'The Count of Monte Cristo',
    '1984': 'Nineteen Eighty-Four',
}

summary = df.groupby('gutenberg_id').agg(
    chunks=('chunk_id', 'count'),
    title=('book_title', 'first'),
    author=('book_author', 'first'),
    avg_chunk_chars=('chunk_size', 'mean'),
    min_chunk_chars=('chunk_size', 'min'),
    max_chunk_chars=('chunk_size', 'max'),
    has_summary=('book_summary', lambda x: (x.astype(str) != '').all()),
).sort_values('chunks', ascending=False)

summary['avg_chunk_chars'] = summary['avg_chunk_chars'].round(0).astype(int)

present = set(df['gutenberg_id'].unique())
missing = set(EXPECTED_BOOKS.keys()) - present
if missing:
    print(f'⚠ MISSING BOOKS: {[EXPECTED_BOOKS[m] for m in missing]}')
else:
    print(f'✓ All {len(EXPECTED_BOOKS)} novels present')

print(f'Total chunks: {len(df):,}')
print()
summary

## 2. Chunk size distribution

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Overall distribution
df['chunk_size'].hist(bins=50, ax=axes[0], edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Chunk size (chars)')
axes[0].set_ylabel('Count')
axes[0].set_title('Chunk Size Distribution (all books)')
axes[0].axvline(2000, color='red', linestyle='--', label='Target: 2000')
axes[0].legend()

# Per-book
df.groupby('book_title')['chunk_size'].mean().sort_values().plot.barh(ax=axes[1])
axes[1].set_xlabel('Mean chunk size (chars)')
axes[1].set_title('Mean Chunk Size by Novel')
axes[1].axvline(2000, color='red', linestyle='--')

plt.tight_layout()
plt.show()

print(f'Chunk size stats: mean={df["chunk_size"].mean():.0f}, '
      f'median={df["chunk_size"].median():.0f}, '
      f'min={df["chunk_size"].min()}, max={df["chunk_size"].max()}')

## 3. Metadata enrichment — book titles, authors, summaries

In [ ]:
for gid in sorted(df['gutenberg_id'].unique(), key=lambda x: df[df['gutenberg_id']==x]['book_title'].iloc[0]):
    row = df[df['gutenberg_id'] == gid].iloc[0]
    title = row['book_title']
    author = row['book_author']
    summary = str(row['book_summary'])
    n_chunks = len(df[df['gutenberg_id'] == gid])
    
    print(f'━━━ {title} by {author} (ID: {gid}, {n_chunks} chunks) ━━━')
    print(f'Summary ({len(summary.split())} words): {summary[:200]}...')
    print()

## 4. Qualitative chunk inspection — first, middle, last chunks per book

Verify that:
- First chunks contain opening text (not Gutenberg boilerplate)
- Middle chunks are coherent narrative passages
- Last chunks contain closing text (not Gutenberg license)

In [ ]:
def show_chunk(row, label=''):
    text = row['article_text']
    print(f'  [{label}] chunk_id={row["chunk_id"]}, {len(text)} chars')
    print(f'  First 300 chars: {repr(text[:300])}')
    print(f'  Last  200 chars: {repr(text[-200:])}')
    print()

for gid in sorted(df['gutenberg_id'].unique(), key=lambda x: df[df['gutenberg_id']==x]['book_title'].iloc[0]):
    book = df[df['gutenberg_id'] == gid].sort_values('chunk_id')
    title = book.iloc[0]['book_title']
    print(f'\n{"═" * 70}')
    print(f'{title} ({len(book)} chunks)')
    print(f'{"═" * 70}')
    
    show_chunk(book.iloc[0], 'FIRST')
    mid = len(book) // 2
    show_chunk(book.iloc[mid], 'MIDDLE')
    show_chunk(book.iloc[-1], 'LAST')

## 5. Overlap verification

Check that consecutive chunks overlap (200 chars expected).

In [ ]:
overlap_checks = []
for gid in df['gutenberg_id'].unique():
    book = df[df['gutenberg_id'] == gid].sort_values('chunk_id')
    texts = book['article_text'].tolist()
    overlaps = []
    for i in range(len(texts) - 1):
        # Check if the end of chunk i appears at the start of chunk i+1
        tail = texts[i][-200:]
        overlap = 0
        for k in range(len(tail), 0, -1):
            if texts[i+1].startswith(tail[-k:]):
                overlap = k
                break
        overlaps.append(overlap)
    title = book.iloc[0]['book_title']
    if overlaps:
        avg = sum(overlaps) / len(overlaps)
        zeros = sum(1 for o in overlaps if o == 0)
        overlap_checks.append({'title': title, 'avg_overlap': round(avg), 'zero_overlaps': zeros, 'pairs': len(overlaps)})

ov_df = pd.DataFrame(overlap_checks).sort_values('title')
print('Overlap between consecutive chunks (expected ~200 chars):')
ov_df

## 6. Prompt preview — what the LLM will see

Show the full user prompt for one sample chunk (book context + chunk text).

In [ ]:
sample = df[df['book_title'] == 'Pride and Prejudice'].iloc[10]

# Reconstruct the prompt as _format_prompt would
title = sample['book_title']
author = sample['book_author']
summary = sample['book_summary']
article_text = sample['article_text']

book_context = f'Novel Context:\nThe text below is a short excerpt from the novel "{title}"'
book_context += f' by {author}'
book_context += (
    '. It is one of many consecutive chunks extracted from the full novel. '
    'The excerpt may begin or end mid-scene. Use the summary below to '
    'understand the broader societal context of the novel when identifying norms.\n'
)
book_context += f'\nNovel summary: {summary}\n\n---\n\n'

prompt = f'{book_context}Text from fiction:\n{article_text}'

print(f'Total prompt length: {len(prompt):,} chars ({len(prompt.split()):,} words)')
print(f'Book context: {len(book_context):,} chars')
print(f'Chunk text: {len(article_text):,} chars')
print()
print('=' * 70)
print('FULL PROMPT PREVIEW')
print('=' * 70)
print(prompt)